In [1]:
library(ggplot2)
library(data.table)
library(stringr)
theme_set(theme_bw())

In [9]:
tools = c('singlem', 'sylph')
d1 = data.table(tool = tools, sample = "SRR8648366")

In [ ]:
readit = function(tool, sample){
    to_read = paste0('output_',tool,'/', tool, '/',sample,'.profile')
    return(fread(to_read))
}
d2 = d1[, readit(tool, sample)[, c("coverage", "taxonomy")], by=list(tool, sample)]
d2[, relabu := coverage / sum(coverage, na.rm = TRUE), by = list(tool, sample)]
d2[, taxonomy := gsub("; ", ";", gsub("Root; ", "", taxonomy))]

In [40]:
m = dcast(d2, sample + taxonomy ~ tool, value.var = c("relabu", "coverage"), fill = 0)
setnames(m, names(m), gsub("_", "__", names(m)))

In [ ]:
format_table <- function(m) {
    ra_cols <- grep("^relabu_|_relabu$", names(m), value = TRUE)
    mtmp = m[which(!grepl("^coverage_|_coverage$", names(m))), with = FALSE]
    mtmp[
        ,
        c(ra_cols, "taxonomy") := c(
            lapply(.SD[, ra_cols, with = FALSE], scales::label_percent()),
            list(purrr::map_chr(strsplit(taxonomy, "; ?"), function(x) tail(x,1)))
        )
    ][]
}

In [ ]:
# kingdom
format_table(
    m[grep("d__", taxonomy),
      lapply(.SD, sum),
      by = list(sample, taxonomy = gsub(";p__.*", "", taxonomy))][relabu__sylph > 0.01]
)

sample,taxonomy,relabu__singlem,relabu__sylph
<chr>,<chr>,<chr>,<chr>
SRR8648366,d__Archaea,1%,1%
SRR8648366,d__Bacteria,99%,99%


In [62]:
# phylum
format_table(
    m[!grepl("d__Archaea", taxonomy) & grepl(";p__", taxonomy),
      lapply(.SD, sum),
      by = list(sample, taxonomy = gsub(";c__.*", "", taxonomy))][relabu__sylph > 0.01]
)

sample,taxonomy,relabu__singlem,relabu__sylph
<chr>,<chr>,<chr>,<chr>
SRR8648366,p__Actinobacteriota,60.7%,59.192%
SRR8648366,p__Bacteroidota,9.1%,8.933%
SRR8648366,p__Firmicutes,2.1%,7.809%
SRR8648366,p__Firmicutes_A,3.1%,10.387%
SRR8648366,p__Proteobacteria,14.6%,10.446%


In [63]:
# class
format_table(
    m[!grepl("d__Archaea", taxonomy) & grepl(";c__", taxonomy),
      lapply(.SD, sum),
      by = list(sample, taxonomy = gsub(";o__.*", "", taxonomy))][relabu__sylph > 0.01]
)

sample,taxonomy,relabu__singlem,relabu__sylph
<chr>,<chr>,<chr>,<chr>
SRR8648366,c__Actinomycetia,57.1%,58.88%
SRR8648366,c__Bacteroidia,8.6%,8.93%
SRR8648366,c__Bacilli,2.1%,7.81%
SRR8648366,c__Clostridia,3.1%,10.39%
SRR8648366,c__Gammaproteobacteria,13.3%,9.52%


In [64]:
# order
format_table(
    m[!grepl("d__Archaea", taxonomy) & grepl(";o__", taxonomy),
      lapply(.SD, sum),
      by = list(sample, taxonomy = gsub(";f__.*", "", taxonomy))][relabu__sylph > 0.01]
)

sample,taxonomy,relabu__singlem,relabu__sylph
<chr>,<chr>,<chr>,<chr>
SRR8648366,o__Actinomycetales,35.51964%,40.2727%
SRR8648366,o__Mycobacteriales,13.11559%,17.2577%
SRR8648366,o__Propionibacteriales,6.08802%,1.0262%
SRR8648366,o__Bacteroidales,1.49753%,7.0323%
SRR8648366,o__Flavobacteriales,4.13192%,1.9010%
SRR8648366,o__Lactobacillales,0.64331%,4.1470%
SRR8648366,o__Staphylococcales,0.14799%,1.2683%
SRR8648366,o__Christensenellales,0.29246%,1.3574%
SRR8648366,o__Clostridiales,0.58894%,2.2603%


In [65]:
# family
format_table(
    m[!grepl("d__Archaea", taxonomy) & grepl(";f__", taxonomy),
      lapply(.SD, sum),
      by = list(sample, taxonomy = gsub(";g__.*", "", taxonomy))][relabu__sylph > 0.01]
)

sample,taxonomy,relabu__singlem,relabu__sylph
<chr>,<chr>,<chr>,<chr>
SRR8648366,f__Brevibacteriaceae,1.17638%,4.8488%
SRR8648366,f__Dermatophilaceae,24.65160%,32.9681%
SRR8648366,f__Mycobacteriaceae,7.82466%,16.3459%
SRR8648366,f__Propionibacteriaceae,2.32079%,1.0262%
SRR8648366,f__Bacteroidaceae,0.26654%,1.6332%
SRR8648366,f__Dysgonomonadaceae,0.49255%,2.6284%
SRR8648366,f__Flavobacteriaceae,2.52692%,1.6350%
SRR8648366,f__Lactobacillaceae,0.21871%,1.3857%
SRR8648366,f__Streptococcaceae,0.31058%,2.4742%


In [66]:
# How well does genus level rescue some of the missing genomes? First need to remake the table with genus level, annoying since kraken profiles are filled, when the rest aren't.
format_table(
    m[!grepl("d__Archaea", taxonomy) & grepl(";g__", taxonomy),
      lapply(.SD, sum),
      by = list(sample, taxonomy = gsub(";s__.*", "", taxonomy))][relabu__sylph > 0.01]
)

sample,taxonomy,relabu__singlem,relabu__sylph
<chr>,<chr>,<chr>,<chr>
SRR8648366,g__Brevibacterium,0.934%,4.8488%
SRR8648366,g__F2B08,2.608%,13.4284%
SRR8648366,g__Ornithinimicrobium,4.644%,18.7970%
SRR8648366,g__Corynebacterium,1.867%,12.7787%
SRR8648366,g__Dietzia,0.579%,1.0383%
SRR8648366,g__Mycobacterium,2.122%,2.1299%
SRR8648366,g__Prevotella,0.194%,1.1673%
SRR8648366,g__Proteiniphilum,0.289%,1.4499%
SRR8648366,g__Aequorivita,0.649%,1.5821%


In [67]:
# Species level
format_table(m[!grepl("d__Archaea", taxonomy) & grepl(";s__", taxonomy)][relabu__sylph > 0.01])

sample,taxonomy,relabu__singlem,relabu__sylph
<chr>,<chr>,<chr>,<chr>
SRR8648366,s__Brevibacterium intestinavium,0.6838%,4.2619%
SRR8648366,s__F2B08 sp012729695,0.1792%,1.1249%
SRR8648366,s__F2B08 sp012838445,2.0117%,12.3035%
SRR8648366,s__Ornithinimicrobium sp003577095,1.4253%,18.7970%
SRR8648366,s__Corynebacterium casei,0.1618%,1.2115%
SRR8648366,s__Corynebacterium humireducens,0.6176%,3.2129%
SRR8648366,s__Corynebacterium pollutisoli,0.2212%,1.5285%
SRR8648366,s__Corynebacterium sp012838715,0.0803%,1.0560%
SRR8648366,s__Corynebacterium sp012838985,0.3758%,5.1265%


In [82]:
m[, grep("s__Corynebacterium", taxonomy, value = TRUE)][1]
m[, grep("s__F2B08", taxonomy, value = TRUE)][1]
m[, grep("s__Ornithinimicrobium", taxonomy, value = TRUE)][1]

[1] "d__Bacteria;p__Actinobacteriota;c__Actinomycetia;o__Mycobacteriales;f__Mycobacteriaceae;g__Corynebacterium;s__Corynebacterium casei"

[1] "d__Bacteria;p__Actinobacteriota;c__Actinomycetia;o__Actinomycetales;f__Dermatophilaceae;g__F2B08;s__F2B08 sp012729695"

[1] "d__Bacteria;p__Actinobacteriota;c__Actinomycetia;o__Actinomycetales;f__Dermatophilaceae;g__Ornithinimicrobium;s__Ornithinimicrobium sp001942405"